> **ARCHIVO LEGADO — no usar como ruta principal.** Se conserva para no perder historial ni resultados. La versión canónica, corregida y con la selección breve de ejercicios es [`03.1.01_Aplicaciones_y_solucion_fundamental_CORREGIDO.ipynb`](03.1.01_Aplicaciones_y_solucion_fundamental_CORREGIDO.ipynb).

# Parte 3 — Ecuaciones elípticas
## 3.1 Las ecuaciones de Poisson y Laplace
### 3.1.01 Aplicaciones iniciales y solución fundamental del laplaciano

**Fuente principal:** página 5 de las notas manuscritas posteriores a la ecuación de onda.

La transcripción conserva la notación y el orden de la página. Toda ampliación aparece
separada como **Aclaración**, **Complemento**, **Corrección editorial** o
**Demostración añadida**.

## Contenido

1. Mecánica de fluidos.
2. Membrana elástica.
3. Movimiento browniano.
4. Invariancia rotacional.
5. Solución fundamental de \(-\Delta\).
6. Identidad \(-\Delta\Phi=\delta_0\).
7. Simulaciones y ejercicios tipo Examen General.

No se desarrollan todavía la propiedad del promedio, el principio del máximo ni la función de Green.

# Simulaciones y visualizaciones

Todas las simulaciones se ejecutan directamente. No es necesario cambiar una bandera
`VIDEO=True`. El notebook detecta CuPy/CUDA automáticamente, guarda sus salidas y las
muestra dentro del propio notebook.

In [ ]:
from __future__ import annotations
import math, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter
from IPython.display import Video, Image, display

BASE = Path("..")
FIG_DIR = BASE / "figuras"
ANIM_DIR = BASE / "animaciones"
FIG_DIR.mkdir(parents=True, exist_ok=True)
ANIM_DIR.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = False
try:
    import cupy as cp
    if cp.cuda.runtime.getDeviceCount() > 0:
        xp = cp
        GPU_AVAILABLE = True
        print("Backend: CuPy/CUDA")
    else:
        raise RuntimeError("No CUDA device")
except Exception as exc:
    xp = np
    print("Backend: NumPy/CPU")
    print("CuPy no disponible:", type(exc).__name__)

def to_cpu(a):
    return cp.asnumpy(a) if GPU_AVAILABLE else np.asarray(a)

def save_show_animation(ani, stem, fps=60, dpi=150, bitrate=12000):
    if shutil.which("ffmpeg"):
        path = ANIM_DIR / f"{stem}.mp4"
        ani.save(path, writer=FFMpegWriter(fps=fps, bitrate=bitrate), dpi=dpi)
        display(Video(str(path), embed=True))
    else:
        path = ANIM_DIR / f"{stem}.gif"
        ani.save(path, writer=PillowWriter(fps=min(fps, 35)), dpi=min(dpi, 110))
        display(Image(filename=str(path)))
    print(path.resolve())
    return path

## Simulación 3.1.A — Flujo potencial incompresible e irrotacional

Tomamos

\[
\phi(x,y)=x^2-y^2,\qquad \mathbf v=\nabla\phi=(2x,-2y).
\]

Entonces

\[
\nabla\cdot\mathbf v=\Delta\phi=0,\qquad \operatorname{curl}\mathbf v=0.
\]

In [ ]:
N = 281
x = np.linspace(-2.0, 2.0, N)
y = np.linspace(-2.0, 2.0, N)
X, Y = np.meshgrid(x, y, indexing="xy")
phi = X**2 - Y**2
vx, vy = 2.0*X, -2.0*Y
dx, dy = x[1]-x[0], y[1]-y[0]
div_v = np.gradient(vx, dx, axis=1) + np.gradient(vy, dy, axis=0)
curl_v = np.gradient(vy, dx, axis=1) - np.gradient(vx, dy, axis=0)
print("||div v||_∞ =", np.max(np.abs(div_v)))
print("||curl v||_∞ =", np.max(np.abs(curl_v)))

fig, ax = plt.subplots(figsize=(9,7))
cs = ax.contour(X, Y, phi, levels=25)
ax.clabel(cs, inline=True, fontsize=7)
s = max(1, N//28)
ax.quiver(X[::s,::s], Y[::s,::s], vx[::s,::s], vy[::s,::s])
ax.set_aspect("equal")
ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$y$")
ax.set_title(r"Flujo potencial $\mathbf v=\nabla(x^2-y^2)$")
fig.tight_layout()
path = FIG_DIR / "03.1.A_flujo_potencial.png"
fig.savefig(path, dpi=220)
plt.show(); plt.close(fig)
print(path.resolve())

## Simulación 3.1.B — Membrana elástica

Se aproxima

\[
-\Delta u=q(x,y)\quad\text{en }(-1,1)^2,\qquad u=0\quad\text{en la frontera},
\]

con una carga localizada. La relajación es un procedimiento numérico para encontrar
el equilibrio, no la dinámica física de la membrana.

In [ ]:
N = 420 if GPU_AVAILABLE else 180
iterations = 2800 if GPU_AVAILABLE else 1200
nframes = 150 if GPU_AVAILABLE else 100
omega = 0.82

xg = xp.linspace(-1.0,1.0,N)
yg = xp.linspace(-1.0,1.0,N)
h = float(to_cpu(xg[1]-xg[0]))
Xg,Yg = xp.meshgrid(xg,yg,indexing="xy")
q = (8*xp.exp(-18*((Xg-.25)**2+(Yg+.15)**2))
     +4*xp.exp(-25*((Xg+.45)**2+(Yg-.35)**2)))
u = xp.zeros_like(q)

save_steps = set(np.unique(np.round(np.geomspace(1,iterations,nframes)).astype(int)).tolist())
snaps=[to_cpu(u).astype(np.float32)]
res=[]

def residual(U):
    R=(4*U[1:-1,1:-1]-U[2:,1:-1]-U[:-2,1:-1]
       -U[1:-1,2:]-U[1:-1,:-2])/h**2-q[1:-1,1:-1]
    return float(to_cpu(xp.max(xp.abs(R))))

res.append(residual(u))
for k in range(1,iterations+1):
    jac=.25*(u[2:,1:-1]+u[:-2,1:-1]+u[1:-1,2:]+u[1:-1,:-2]
             +h*h*q[1:-1,1:-1])
    un=u.copy()
    un[1:-1,1:-1]=(1-omega)*u[1:-1,1:-1]+omega*jac
    u=un
    if k in save_steps:
        snaps.append(to_cpu(u).astype(np.float32))
        res.append(residual(u))

snaps=np.asarray(snaps)
print("Residuo inicial:",res[0],"Residuo final:",res[-1])

fig,ax=plt.subplots(figsize=(10,7))
im=ax.imshow(snaps[0],origin="lower",extent=[-1,1,-1,1],
             interpolation="bilinear",vmin=0,vmax=float(snaps[-1].max()))
fig.colorbar(im,ax=ax,label=r"$u(x,y)$")
ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$y$")
ax.set_title("Relajación de una membrana elástica")
txt=ax.text(.02,.98,"",transform=ax.transAxes,va="top",
            bbox={"boxstyle":"round","alpha":.8})
def upd(i):
    im.set_data(snaps[i])
    txt.set_text(f"fotograma {i+1}/{len(snaps)}\n"
                 +rf"$\|-\Delta_hu-q\|_\infty={res[i]:.3e}$")
    return im,txt
ani=FuncAnimation(fig,upd,frames=len(snaps),interval=1000/60,blit=False)
fig.tight_layout()
save_show_animation(ani,"03.1.B_membrana_relajacion",
                    fps=120 if GPU_AVAILABLE else 60,
                    dpi=180 if GPU_AVAILABLE else 140,
                    bitrate=18000 if GPU_AVAILABLE else 9000)
plt.close(fig)

Uc=snaps[-1]
Xc,Yc=np.meshgrid(to_cpu(xg),to_cpu(yg),indexing="xy")
fig=plt.figure(figsize=(10,7))
ax=fig.add_subplot(111,projection="3d")
sk=3 if GPU_AVAILABLE else 2
ax.plot_surface(Xc[::sk,::sk],Yc[::sk,::sk],Uc[::sk,::sk],
                linewidth=0,antialiased=True)
ax.set_xlabel(r"$x$"); ax.set_ylabel(r"$y$"); ax.set_zlabel(r"$u$")
ax.set_title("Configuración de equilibrio")
fig.tight_layout()
path=FIG_DIR/"03.1.B_membrana_equilibrio.png"
fig.savefig(path,dpi=220)
plt.show(); plt.close(fig)
print(path.resolve())

## Simulación 3.1.C — Movimiento browniano y medida armónica

Para el anillo \(r_1<|x|<r_2\), la probabilidad de alcanzar primero la frontera exterior es

\[
u(r)=\frac{\log(r/r_1)}{\log(r_2/r_1)}.
\]

Se compara esta solución armónica con una simulación Monte Carlo.

In [ ]:
rng = xp.random.default_rng(20260718) if GPU_AVAILABLE else np.random.default_rng(20260718)
r1,r2=.55,1.55
radii=np.linspace(r1+.08,r2-.08,11)
ntrials=40000 if GPU_AVAILABLE else 7000
step=.025
max_steps=25000
est=[]; se=[]

for r0 in radii:
    pos=xp.zeros((ntrials,2),dtype=xp.float32)
    pos[:,0]=r0
    alive=xp.ones(ntrials,dtype=bool)
    outer_hit=xp.zeros(ntrials,dtype=bool)
    for _ in range(max_steps):
        count=int(to_cpu(xp.sum(alive)))
        if count==0:
            break
        idx=xp.where(alive)[0]
        ang=2*math.pi*rng.random(count)
        pos[idx,0]+=step*xp.cos(ang)
        pos[idx,1]+=step*xp.sin(ang)
        rr=xp.sqrt(pos[idx,0]**2+pos[idx,1]**2)
        outer=rr>=r2
        inner=rr<=r1
        done=outer|inner
        if int(to_cpu(xp.sum(outer)))>0:
            outer_hit[idx[outer]]=True
        alive[idx[done]]=False
    p=float(to_cpu(xp.mean(outer_hit)))
    est.append(p)
    se.append(math.sqrt(max(p*(1-p),0)/ntrials))

est=np.asarray(est); se=np.asarray(se)
exact=np.log(radii/r1)/np.log(r2/r1)
rr=np.linspace(r1,r2,500)

fig,ax=plt.subplots(figsize=(9,6))
ax.plot(rr,np.log(rr/r1)/np.log(r2/r1),label="solución armónica exacta")
ax.errorbar(radii,est,yerr=2*se,fmt="o",capsize=4,label="Monte Carlo")
ax.set_xlabel(r"$r$")
ax.set_ylabel("probabilidad de salida por la frontera exterior")
ax.set_ylim(-.04,1.04)
ax.grid(True,alpha=.3); ax.legend()
ax.set_title("Movimiento browniano y problema de Dirichlet")
fig.tight_layout()
path=FIG_DIR/"03.1.C_browniano_medida_armonica.png"
fig.savefig(path,dpi=220)
plt.show(); plt.close(fig)
print("Error máximo:",np.max(np.abs(est-exact)))
print(path.resolve())

## Simulación 3.1.D — Concentración distribucional en el polo

En dimensión dos,

\[
\Phi(x)=-\frac1{2\pi}\log|x|.
\]

La regularización

\[
\Phi_\varepsilon(x)=-\frac1{4\pi}\log(|x|^2+\varepsilon^2)
\]

satisface

\[
-\Delta\Phi_\varepsilon(x)
=\frac{\varepsilon^2}{\pi(|x|^2+\varepsilon^2)^2}.
\]

Se verifica numéricamente que esta densidad tiene masa próxima a uno y que
su acción sobre una función de prueba converge a la evaluación en el origen.

In [ ]:
N=760 if GPU_AVAILABLE else 460
L=4.0
x=xp.linspace(-L,L,N)
y=xp.linspace(-L,L,N)
dx=float(to_cpu(x[1]-x[0]))
X,Y=xp.meshgrid(x,y,indexing="xy")
R2=X**2+Y**2
psi=xp.exp(-(X**2+2*Y**2))*xp.cos(.7*X+.3*Y)
eps_values=np.geomspace(.8,.04,24)
masses=[]; pairings=[]
for eps in eps_values:
    rho=eps**2/(math.pi*(R2+eps**2)**2)
    masses.append(float(to_cpu(xp.sum(rho)*dx*dx)))
    pairings.append(float(to_cpu(xp.sum(rho*psi)*dx*dx)))
masses=np.asarray(masses); pairings=np.asarray(pairings)

fig,ax=plt.subplots(figsize=(9,6))
ax.semilogx(eps_values,masses,"o-",label=r"$\int\rho_\varepsilon$")
ax.semilogx(eps_values,pairings,"s-",label=r"$\int\rho_\varepsilon\psi$")
ax.axhline(1,linestyle="--",label=r"$\psi(0)=1$")
ax.invert_xaxis()
ax.set_xlabel(r"$\varepsilon$"); ax.set_ylabel("valor")
ax.grid(True,alpha=.3); ax.legend()
ax.set_title(r"Concentración de $-\Delta\Phi_\varepsilon$")
fig.tight_layout()
path=FIG_DIR/"03.1.D_delta_distribucional.png"
fig.savefig(path,dpi=220)
plt.show(); plt.close(fig)
print("Masa final:",masses[-1])
print("Emparejamiento final:",pairings[-1])
print(path.resolve())

# Transcripción numerada de las notas

## 3.1.1 Mecánica de fluidos

> **Transcripción de las notas.**
>
> **I) Mecánica de fluidos. (Descripción euleriana).**
>
> \[
> \vec u=(u_1,u_2,u_3)\in\mathbb R^3
> \]
>
> campo de velocidades estacionario:
>
> \[
> \begin{cases}
> \nabla\cdot\vec u=0, & \text{incompresible},\\
> \nabla\times\vec u=0, & \text{irrotacional}.
> \end{cases}
> \]
>
> \[
> \vec u=\nabla\phi
> \quad\Longrightarrow\quad
> \nabla\cdot\vec u=0
> \quad\Longrightarrow\quad
> \Delta\phi=0.
> \]

### **Definición 3.1.1 (Campo incompresible).**

Sea \(\Omega\subset\mathbb R^3\) abierto y sea
\(\vec u\in C^1(\Omega;\mathbb R^3)\). Se dice que \(\vec u\) es
**incompresible** en \(\Omega\) si

\[
\nabla\cdot\vec u=0.
\]

### **Definición 3.1.2 (Campo irrotacional).**

Bajo las mismas hipótesis, \(\vec u\) es **irrotacional** si

\[
\nabla\times\vec u=0.
\]

### **Proposición 3.1.3 (Potencial de un flujo incompresible e irrotacional).**

Sea \(\Omega\subset\mathbb R^3\) un dominio simplemente conexo y
\(\vec u\in C^1(\Omega;\mathbb R^3)\). Si \(\nabla\times\vec u=0\),
existe \(\phi\in C^2(\Omega)\) tal que \(\vec u=\nabla\phi\).
Si además \(\nabla\cdot\vec u=0\), entonces

\[
\Delta\phi=0\quad\text{en }\Omega.
\]

> **Aclaración.** La simple conexidad no aparece escrita en la página, pero es necesaria
> para garantizar globalmente la existencia del potencial.

### Ejercicios — Sección 3.1.1

1. Para \(\phi(x,y,z)=x^2-y^2\), calcule \(\vec u=\nabla\phi\) y verifique
   incompresibilidad e irrotacionalidad.

2. En
   \[
   \Omega=\mathbb R^3\setminus\{(0,0,z):z\in\mathbb R\},
   \]
   estudie
   \[
   \vec u(x,y,z)=\left(-\frac y{x^2+y^2},\frac x{x^2+y^2},0\right).
   \]
   Explique por qué la irrotacionalidad no produce un potencial global monovaluado.

3. **Tipo Examen General.** Sea \(\Omega\subset\mathbb R^3\) simplemente conexo,
   acotado y suave. Si
   \[
   \nabla\cdot\vec u=0,\qquad \nabla\times\vec u=0,\qquad
   \vec u\cdot\nu=0\text{ en }\partial\Omega,
   \]
   demuestre por energía que \(\vec u\equiv0\).

## 3.1.2 Membrana elástica

> **Transcripción de las notas.**
>
> **II) Membrana elástica.**
>
> \[
> E\simeq
> \int_\Omega \frac{k}{2}|\nabla u|^2\,dx
> -
> \int_\Omega q\,\rho(x)u(x)\,dx.
> \]
>
> \[
> \Longrightarrow
> \begin{cases}
> -\Delta u=\dfrac qk\,\rho(x),&x\in\Omega,\\
> u=0,&x\in\partial\Omega.
> \end{cases}
> \]

### **Definición 3.1.4 (Funcional de energía de la membrana).**

Sean \(\Omega\subset\mathbb R^2\) un dominio acotado, \(k>0\) y
\(f\in L^2(\Omega)\). Se define

\[
\mathcal E[v]=\frac{k}{2}\int_\Omega|\nabla v|^2\,dx-\int_\Omega fv\,dx,
\qquad v\in H_0^1(\Omega).
\]

### **Proposición 3.1.5 (Ecuación de Euler–Lagrange de la membrana).**

Si \(u\in H_0^1(\Omega)\) minimiza \(\mathcal E\), entonces

\[
k\int_\Omega\nabla u\cdot\nabla\varphi\,dx
=
\int_\Omega f\varphi\,dx
\]

para toda \(\varphi\in H_0^1(\Omega)\). Si además
\(u\in C^2(\Omega)\cap C(\overline\Omega)\), entonces

\[
-k\Delta u=f\quad\text{en }\Omega,\qquad
u=0\quad\text{en }\partial\Omega.
\]

> **Complemento.** En las notas \(f=q\rho\).

### Ejercicios — Sección 3.1.2

1. Calcule la primera variación de \(\mathcal E[u+\varepsilon\varphi]\).

2. Para \(\Omega=B_R(0)\subset\mathbb R^2\) y carga constante \(f_0>0\),
   encuentre una solución radial.

3. **Tipo Examen General.** Si
   \[
   -\Delta u=f,\qquad u|_{\partial\Omega}=0,
   \]
   demuestre
   \[
   \int_\Omega|\nabla u|^2\,dx=\int_\Omega fu\,dx
   \]
   y úsela para probar unicidad.

## 3.1.3 Movimiento browniano

> **Transcripción de las notas.**
>
> **III) Movimiento Browniano.**
>
> Caminata aleatoria simétrica y Markoviana.
>
> \(u(x)\): probabilidad de que la partícula, iniciando en \(x\in\Omega\),
> termine en una componente de la frontera.
>
> \[
> \begin{cases}
> u=u(x),&x\in\Omega,\\
> u=0,&\text{en }\Gamma_1,\\
> u=1,&\text{en }\Gamma_2.
> \end{cases}
> \]
>
> \[
> \Longrightarrow\quad\Delta u=0\quad\text{en }\Omega.
> \]

> **Aclaración de lectura.** La figura muestra
> \(\partial\Omega=\Gamma_1\cup\Gamma_2\).

### **Definición 3.1.6 (Probabilidad de salida o medida armónica).**

Sea \(\Omega\subset\mathbb R^n\) acotado y sea \(A\subset\partial\Omega\).
Para un movimiento browniano \(B_t\) iniciado en \(x\), defínase

\[
\tau_\Omega=\inf\{t>0:B_t\notin\Omega\}.
\]

La función

\[
u(x)=\mathbb P_x(B_{\tau_\Omega}\in A)
\]

es la probabilidad de salida por \(A\). Bajo hipótesis regulares, resuelve el
problema de Dirichlet con dato indicador de \(A\).

### Ejercicios — Sección 3.1.3

1. Resuelva el problema radial en el anillo de dimensión dos con datos \(0\) y \(1\)
   en las dos componentes de frontera.

2. Repita el problema en dimensión \(n\ge3\).

3. **Tipo Examen General.** Interprete probabilísticamente la solución y justifique
   directamente que \(0\le u\le1\), sin usar todavía el principio del máximo.

## 3.1.4 Invariancia rotacional

> **Transcripción de las notas.**
>
> La ecuación de Laplace es invariante bajo rotaciones.

### **Proposición 3.1.7 (Invariancia del laplaciano bajo rotaciones).**

Sea \(\Omega\subset\mathbb R^n\) abierto, \(Q\in O(n)\) y
\(u\in C^2(\Omega)\). Defínase

\[
v(x)=u(Qx),\qquad x\in Q^T\Omega.
\]

Entonces

\[
\Delta v(x)=(\Delta u)(Qx).
\]

En particular, si \(u\) es armónica, \(v\) también lo es.

### Demostración

\[
\nabla v(x)=Q^T\nabla u(Qx),\qquad
D^2v(x)=Q^TD^2u(Qx)Q.
\]

Como \(\Delta v=\operatorname{tr}D^2v\),

\[
\Delta v(x)
=
\operatorname{tr}(Q^TD^2u(Qx)Q)
=
\operatorname{tr}(D^2u(Qx)QQ^T)
=
(\Delta u)(Qx).
\]

\(\square\)

### Ejercicios — Sección 3.1.4

1. Verifique el resultado directamente en dimensión dos usando \(Q_\theta\).

2. Demuestre la invariancia de Poisson si \(f_Q(x)=f(Qx)\).

3. Determine qué matrices constantes \(A\) hacen que
   \[
   Lu=-\operatorname{tr}(AD^2u)
   \]
   sea invariante bajo todas las rotaciones.

## 3.1.5 Solución fundamental del laplaciano

> **Transcripción de las notas.**
>
> **Sol. Fund. de \(-\Delta\).**
>
> \[
> \Phi(x)=
> \begin{cases}
> -\dfrac{1}{2\pi}\log|x|,&n=2,\\[2mm]
> \dfrac{1}{(n-2)\omega_n|x|^{n-2}},&n\ge3.
> \end{cases}
> \]
>
> \[
> \omega_n=\frac{2\pi^{n/2}}{\Gamma(n/2)},\qquad
> \Gamma(s)=\int_0^\infty e^{-y}y^{s-1}\,dy.
> \]
>
> \[
> |\partial B_1|=\omega_n,\qquad |B_1|=\frac{\omega_n}{n}.
> \]

### **Definición 3.1.8 (Solución fundamental de \(-\Delta\)).**

Una distribución \(\Phi\in\mathcal D'(\mathbb R^n)\) es una
**solución fundamental de \(-\Delta\)** si

\[
-\Delta\Phi=\delta_0
\]

en \(\mathcal D'(\mathbb R^n)\).

### **Proposición 3.1.9 (Integrabilidad local de \(\Phi\)).**

La función anterior pertenece a \(L^1_{\mathrm{loc}}(\mathbb R^n)\). Para \(0<r\le1\),

\[
\int_{B_r}|\Phi(x)|\,dx
\le
\begin{cases}
Cr^2(1+|\log r|),&n=2,\\
Cr^2,&n\ge3,
\end{cases}
\]

con \(C=C(n)\).

> **Corrección editorial.** La estimación manuscrita en dimensión dos aparece abreviada.
> Se presenta una forma válida uniformemente para \(0<r\le1\).

## **Teorema 3.1.10 (Solución fundamental del laplaciano).**

Sea \(n\ge2\) y sea \(\Phi\) la función anterior. Para cada \(y\in\mathbb R^n\):

1. 
   \[
   \Delta_x\Phi(x-y)=0\qquad(x\neq y);
   \]

2. para toda \(\varphi\in C_c^\infty(\mathbb R^n)\),
   \[
   -\int_{\mathbb R^n}\Phi(x-y)\Delta\varphi(x)\,dx
   =
   \varphi(y).
   \]

Por tanto,

\[
-\Delta_x\Phi(x-y)=\delta_y
\]

en el sentido de distribuciones.

### Demostración añadida

La primera afirmación se obtiene con

\[
\Delta f(r)=f''(r)+\frac{n-1}{r}f'(r).
\]

Para la identidad distribucional se aplica la segunda identidad de Green en un
dominio perforado por \(B_\varepsilon(y)\). La normal exterior del dominio perforado
apunta hacia el polo. La normalización de \(\Phi\) da

\[
-\frac{\partial\Phi}{\partial\nu}
=
\frac1{\omega_n\varepsilon^{n-1}}
\quad\text{en }\partial B_\varepsilon(y).
\]

Entonces

\[
\frac1{\omega_n\varepsilon^{n-1}}
\int_{\partial B_\varepsilon(y)}\varphi\,dS
\longrightarrow\varphi(y).
\]

El otro término de frontera tiende a cero por la integrabilidad local de \(\Phi\).
Al hacer \(\varepsilon\downarrow0\) se obtiene la identidad. \(\square\)

### Ejercicios — Sección 3.1.5

1. Verifique que \(\Delta\log|x|=0\) fuera del origen en dimensión dos y que
   \(\Delta|x|^{2-n}=0\) fuera del origen para \(n\ge3\).

2. Demuestre la integrabilidad local de \(\Phi\).

3. Calcule
   \[
   \int_{\partial B_r}\frac{\partial\Phi}{\partial\nu}\,dS
   \]
   y explique cómo fija la constante de normalización.

4. **Tipo Examen General.** Para \(L=\Delta-c^2\) en \(\mathbb R^3\), \(c>0\),
   encuentre una solución fundamental radial y determine su constante por flujo.

5. **Tipo Examen General.** Demuestre que
   \[
   v(x)=\frac1{8\pi}|x|^2\log|x|
   \]
   es una solución fundamental de \(\Delta^2\) en \(\mathbb R^2\), precisando el
   sentido distribucional.

# Control de cobertura del temario

## Cubierto en esta unidad

- mecánica de fluidos;
- membrana elástica;
- movimiento browniano;
- invariancia rotacional;
- fórmula de la solución fundamental;
- integrabilidad local;
- armonicidad fuera del polo;
- identidad distribucional.

## Añadido para cerrar hipótesis y pasos

- simple conexidad para el potencial global;
- formulación variacional;
- definición precisa de probabilidad de salida;
- demostración de invariancia rotacional;
- definición distribucional;
- demostración mediante dominio perforado.

## Pendiente inmediato

La página 6 comienza con representación mediante \(\Phi\), propiedad del promedio,
subarmonicidad e inicio del principio del máximo. Esos temas corresponden al siguiente notebook.

# Prompt subsecuente

> Continúa exclusivamente con la página 6 de las notas manuscritas. Transcribe
> literalmente, conserva la notación y numera desde 3.2.1. Desarrolla la representación
> mediante la solución fundamental, la propiedad del promedio sobre esferas y bolas y
> las definiciones de subarmonicidad. Marca por separado toda hipótesis agregada. Incluye
> simulaciones automáticas que comparen promedios para funciones armónicas y no armónicas.
> Cierra cada sección con ejercicios nuevos de nivel Examen General. No avances a la página 7.